[Guide Index](README.md)


# 02b Inferencing using RDF/RDFS/OWL-RL


Inferencing is the process of deriving new information from existing graph information according to a specified set of rules, semantics, or expressions.  Inferencing creates new triples based on existing triples in the graph.

StarLayer supports inferencing in a variety of areas.
- rdf/rdfs/owl-rl inferencing (this guide)
- owl 2 dl reasoning ([OWL 2 DL Reasoning guide](02c-owl-dl-reasoning.ipynb))
- shacl rules ([SHACL inference rules](04c-shacl-inference-rules.ipynb))
- sparql rules langiage ([SRL (SPARQL Rules) guide](03b-srl-sparql-rules.ipynb))

`StarLayerGraph` materializes entailments with `infer()`. Applying an entailment regime directly at query time, via `query(entailment=...)`, without materializing a closure first, is covered by the [SPARQL entailment guide](03a-sparql-entailment.ipynb). This guide covers `infer()`-based materialization only: `infer()` materializes the closure, which you then query.

A `StarLayerGraph` `infer()` call with no `entailment` uses **Simple Entailment** - plain graph pattern matching. The SPARQL 1.2 [Entailment Regimes](https://www.w3.org/TR/sparql12-entailment/) specification defines six regimes beyond that baseline. StarLayer provides partial support, as outlined below.

| # | Regime | Status |
|---|---|---|
| 1 | RDF Entailment | ✅ `infer(entailment=ENTAILMENT.RDF)` (section 2 below)|
| 2 | RDFS Entailment | ✅ `infer(entailment=ENTAILMENT.RDFS)` (section 3 below) |
| 3 | D-Entailment (datatype canonicalization) | ❌ Not supported |
| 4 | OWL 2 RDF-Based Semantics | ✅ Done for OWL 2 RL, the decidable rule-based fragment - `infer(entailment=ENTAILMENT['OWL-RDF-Based'])` (section 4 below). ❌ Not supported for the full regime - undecidable in general |
| 5 | OWL 2 Direct Semantics | ✅ Done - genuine OWL 2 DL reasoning via HermiT (default) or RustDL - see the [OWL 2 DL reasoning guide](02c-owl-dl-reasoning.ipynb) |
| 6 | RIF Core Entailment | ❌ Not supported |

`owlrl` is a Python package that implements rdfs/owl-rl reasoning, on rdflib graphs. The wrapper integrates it through `StarLayerGraph.infer()`: the RDF regime runs natively on Oxigraph terms, and the RDFS and OWL-RL regimes copy the graph into an rdflib graph, run `owlrl` there to a fixpoint, and copy only the new triples back as Oxigraph terms. What you hand in and get back is always a `StarLayerGraph`.

This guide provides a brief overview of inferencing with `StarLayerGraph`, via `infer()`.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier sections.


In [1]:
from starlayer.graph import ENTAILMENT
from starlayer.graph.oxgraph import StarLayerGraph

EX = "http://example.org/"

def show(graph, label="Resulting graph", also_ns=None):
    """Helper function to serialize a graph as Turtle 1.2 - used throughout this
    guide to show a graph's state before/after inferencing.

    Filtered to ex: subjects only. infer() adds a series of RDFS/OWL/XSD
    axiomatic vocabulary triples (rdfs:Resource typing, owl:sameAs reflexivity
    for every built-in class/datatype, etc.) Excluding these triples avoids
    unneeded complexity to the example output.

    also_ns: an additional namespace to include, matched against the predicate or
    object - useful for a triple whose subject isn't ex: but is still worth showing
    (e.g. an owlrl inconsistency triple, whose subject is a synthetic BNode).

    run print(g.serialize(format="turtle", pretty=True)) to see the complete graph.
    """
    filtered = StarLayerGraph()
    filtered.bind("ex", EX)
    filtered.bind("rdf", "http://www.w3.org/1999/02/22-rdf-syntax-ns#")
    filtered.bind("rdfs", "http://www.w3.org/2000/01/rdf-schema#")
    filtered.bind("owl", "http://www.w3.org/2002/07/owl#")
    if also_ns is not None:
        filtered.bind("err", also_ns)
    # terms are pyoxigraph terms: .value is the bare IRI (or label), str() is the N-Triples form
    value = lambda t: getattr(t, "value", "")
    for s, p, o in graph:
        also = also_ns is not None and (value(p).startswith(also_ns) or value(o).startswith(also_ns))
        if value(s).startswith(EX) or also:
            filtered.add((s, p, o))
    print(f"\n{label}")
    print(filtered.serialize(format="turtle", pretty=True, only_used=True))

## 1. How `infer()` works

`StarLayerGraph.infer(entailment=..., mode=...)` has two independent parameters:

- **`entailment`** — controls which ruleset to use in inferencing: any one of `ENTAILMENT.RDF`, `ENTAILMENT.RDFS`, `ENTAILMENT['OWL-RDF-Based']`, or a combination of more than one (e.g. `{ENTAILMENT.RDFS, ENTAILMENT['OWL-RDF-Based']}`, or all three at once). (The `ENTAILMENT` IRIs are the W3C regime IRIs; the wrapper also accepts them as `str` or `NamedNode`.)
- **`mode`** — controls the output: `"full"` (default, a **new** graph with the original triples plus everything entailed, original untouched), `"delta"` (a **new** graph with just the newly-entailed triples), or `"in-place"` (mutated original graph). All three ultimately reach the same *closure* (original data plus everything entailed from it) just in different graphs.

Note: with `mode="in-place"`, subsequent changes to the graph do not delete any entailed triples an earlier `infer()` call may have already added.

The `engine=` argument (HermiT, RustDL) belongs to OWL 2 Direct Semantics and is covered in the [OWL 2 DL reasoning guide](02c-owl-dl-reasoning.ipynb).

## 2. RDF reasoning

`entailment=ENTAILMENT.RDF` is the narrowest regime

- **rdfD2** - every predicate actually used anywhere in the graph is entailed `rdf:type rdf:Property`.
- **RDF axiomatic triples** - a few built-in RDF terms (rdf:type, list/container vocabulary like rdf:first/rdf:_1) that the spec always types as rdf:Property/rdf:List.
- Datatype literal value-space well-typing (rdfD1/rdfD1a) is deliberately excluded.

`ENTAILMENT.RDF` can combines with `ENTAILMENT.RDFS`/`ENTAILMENT['OWL-RDF-Based']`, but in practice rarely adds anything those don't already cover. Included for completeness.

In [2]:
g_rdf = StarLayerGraph()
g_rdf.bind("ex", EX)
g_rdf.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:worksAt ex:Acme .
""", format="turtle")

show(g_rdf, "Before")

closed = g_rdf.infer(entailment=ENTAILMENT.RDF)

show(closed, "After infer(entailment=ENTAILMENT.RDF)")


Before
@prefix ex: <http://example.org/> .

ex:alice ex:worksAt ex:Acme .


After infer(entailment=ENTAILMENT.RDF)
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:alice ex:worksAt ex:Acme .

ex:worksAt a rdf:Property .



## 3. RDFS reasoning

`g.infer(entailment=ENTAILMENT.RDFS)` or `g.infer(entailment=ENTAILMENT.RDFS, mode="full")` materializes RDFS entailments into a new graph. `g` itself is left untouched. `entailment=ENTAILMENT.RDFS` covers the full RDFS "data" ruleset - four separate rules:

- **`rdfs:subClassOf`** (rdfs9) - a class's declared superclasses apply to `rdf:type` queries across any number of hops.
- **`rdfs:domain`** (rdfs2) - a property's declared domain entails a type for its subject.
- **`rdfs:range`** (rdfs3) - a property's declared range entails a type for its object.
- **`rdfs:subPropertyOf`** (rdfs5/rdfs7) - itself reflexive and transitive, so a query for a general relation also matches its more specific sub-relations.

`entailment=ENTAILMENT.RDFS` deliberately excludes RDFS's remaining "vocabulary-level" rules - universal `rdfs:Resource` typing (rdfs4a/4b) and the fixed RDFS axiomatic triples - since they don't answer a real question about the data. The same goes for the two RDF 1.2 rules about triple terms: rdfs14 (a triple term entails `_:n rdf:type rdfs:Proposition` for a blank node `_:n` that stands for it) and rdfs14a (every graph entails that some blank node is an `rdfs:Proposition`). They only assert that an anonymous node exists, so materializing them would add a blank node per triple term and nothing a query could use: the SPARQL Entailment Regimes draft says blank nodes introduced by saturation are not returned in solutions anyway.

The example below shows `rdfs:subClassOf` transitivity (rdfs9).

The closure printed below does contain `a rdfs:Resource` typings for the terms in the data, which `owlrl` derives; what the regime leaves out is the fixed set of RDFS axiomatic vocabulary triples.

In [3]:
# alice is a manager which means she is also an employee
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:alice a ex:Manager .
""", format="turtle")

show(g, "Before")

closed = g.infer(entailment=ENTAILMENT.RDFS)

show(closed, "After infer(entailment=ENTAILMENT.RDFS)")
print("original graph untouched:", len(g), "triples")


Before
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Manager rdfs:subClassOf ex:Employee .

ex:alice a ex:Manager .


After infer(entailment=ENTAILMENT.RDFS)
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Employee a rdfs:Resource .

ex:Manager a rdfs:Resource ;
    rdfs:subClassOf ex:Employee .

ex:alice a ex:Employee, ex:Manager, rdfs:Resource .

original graph untouched: 2 triples


### 3.2 `rdfs:domain`, `rdfs:range`, and `rdfs:subPropertyOf`

The same `entailment=ENTAILMENT.RDFS` value also covers the rest of the RDFS "data" ruleset - a property's declared `rdfs:domain`/`rdfs:range` entails a type for its subject/object, and `rdfs:subPropertyOf` lets a query for a general relation also match its more specific sub-relations. 

In [4]:
# ex:manages declares a domain, a range, and a subPropertyOf - bob manages
# alice entails bob is a Manager (domain), alice is an Employee (range),
# and bob worksWith alice (subPropertyOf).
g_rdfs2 = StarLayerGraph()
g_rdfs2.bind("ex", EX)
g_rdfs2.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:manages rdfs:domain ex:Manager ;
        rdfs:range ex:Employee ;
        rdfs:subPropertyOf ex:worksWith .
    ex:bob ex:manages ex:alice .
""", format="turtle")

show(g_rdfs2, "Before")

closed_rdfs2 = g_rdfs2.infer(entailment=ENTAILMENT.RDFS)

show(closed_rdfs2, "After infer(entailment=ENTAILMENT.RDFS)")


Before
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:bob ex:manages ex:alice .

ex:manages rdfs:domain ex:Manager ;
    rdfs:range ex:Employee ;
    rdfs:subPropertyOf ex:worksWith .


After infer(entailment=ENTAILMENT.RDFS)
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Employee a rdfs:Resource .

ex:Manager a rdfs:Resource .

ex:alice a ex:Employee, rdfs:Resource .

ex:bob a ex:Manager, rdfs:Resource ;
    ex:manages ex:alice ;
    ex:worksWith ex:alice .

ex:manages a rdf:Property, rdfs:Resource ;
    rdfs:domain ex:Manager ;
    rdfs:range ex:Employee ;
    rdfs:subPropertyOf ex:manages, ex:worksWith .

ex:worksWith a rdf:Property, rdfs:Resource ;
    rdfs:subPropertyOf ex:worksWith .



## 4. OWL 2 RL reasoning

`g.infer(entailment=ENTAILMENT['OWL-RDF-Based'])` runs a similar forward-chaining closure, but over the OWL 2 RL rule set, producing entailments RDFS alone does not, such as those from `owl:equivalentClass`, `owl:sameAs`, and property characteristics like `owl:TransitiveProperty`.

### 4.1 `owl:equivalentClass`

Two systems label the same role differently — HR calls it `ex:Manager`, the CRM calls the same thing `ex:TeamLead`. Declaring them `owl:equivalentClass` lets an instance of one be recognized as the other.

In [5]:
# alice is a manager, but she is also a teamlead, because manager and teamlead are the same
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager owl:equivalentClass ex:TeamLead .
    ex:alice a ex:Manager .
    ex:Manager rdfs:subClassOf ex:Employee .

""", format="turtle")

show(g, "Before")

closed = g.infer(entailment=ENTAILMENT['OWL-RDF-Based'])

show(closed, "After infer(entailment=ENTAILMENT['OWL-RDF-Based'])")


Before
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Manager rdfs:subClassOf ex:Employee ;
    owl:equivalentClass ex:TeamLead .

ex:alice a ex:Manager .


After infer(entailment=ENTAILMENT['OWL-RDF-Based'])
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Employee owl:sameAs ex:Employee .

ex:Manager rdfs:subClassOf ex:Employee, ex:TeamLead ;
    owl:equivalentClass ex:TeamLead ;
    owl:sameAs ex:Manager .

ex:TeamLead rdfs:subClassOf ex:Employee, ex:Manager ;
    owl:equivalentClass ex:Manager ;
    owl:sameAs ex:TeamLead .

ex:alice a ex:Employee, ex:Manager, ex:TeamLead ;
    owl:sameAs ex:alice .



### 4.2 Property characteristics: `owl:TransitiveProperty`, `owl:SymmetricProperty`, `owl:inverseOf`

Declaring `ex:partOf` to be an `owl:TransitiveProperty` lets a chain of `partOf` facts entail the full transitive closure, not just each direct link. `owl:SymmetricProperty` entails the reverse direction of every asserted fact; `owl:inverseOf` entails a second property's facts from the first's.

In [6]:
# the SalesTeam is part of the SalesDepartment, which means the SalesTeam is a part of AcmeCorp (transitive).
# bob is friendOf alice is entailed via the symmetric property.
# dave is managedBy carol is entailed via the inverse property.
g_props = StarLayerGraph()
g_props.bind("ex", EX)
g_props.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .

    ex:partOf a owl:TransitiveProperty .
    ex:SalesTeam ex:partOf ex:SalesDept .
    ex:SalesDept ex:partOf ex:AcmeCorp .

    ex:friendOf a owl:SymmetricProperty .
    ex:alice ex:friendOf ex:bob .

    ex:manages owl:inverseOf ex:managedBy .
    ex:carol ex:manages ex:dave .
""", format="turtle")

show(g_props, "Before")

closed_props = g_props.infer(entailment=ENTAILMENT['OWL-RDF-Based'])

show(closed_props, "After infer(entailment=ENTAILMENT['OWL-RDF-Based'])")


Before
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:SalesDept ex:partOf ex:AcmeCorp .

ex:SalesTeam ex:partOf ex:SalesDept .

ex:alice ex:friendOf ex:bob .

ex:carol ex:manages ex:dave .

ex:friendOf a owl:SymmetricProperty .

ex:manages owl:inverseOf ex:managedBy .

ex:partOf a owl:TransitiveProperty .


After infer(entailment=ENTAILMENT['OWL-RDF-Based'])
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:AcmeCorp owl:sameAs ex:AcmeCorp .

ex:SalesDept ex:partOf ex:AcmeCorp ;
    owl:sameAs ex:SalesDept .

ex:SalesTeam ex:partOf ex:AcmeCorp, ex:SalesDept ;
    owl:sameAs ex:SalesTeam .

ex:alice ex:friendOf ex:bob ;
    owl:sameAs ex:alice .

ex:bob ex:friendOf ex:alice ;
    owl:sameAs ex:bob .

ex:carol ex:manages ex:dave ;
    owl:sameAs ex:carol .

ex:dave ex:managedBy ex:carol ;
    owl:sameAs ex:dave .

ex:friendOf a owl:SymmetricProperty ;
    owl:sameAs ex:friendOf .

ex:managedBy owl:sameAs ex:ma

### 4.3 `owl:sameAs` — congruence closure

Alice Smith and Alice Jones are declared to be the same entity (`owl:sameAs`) - every fact known about one is entailed for the other.

In [7]:
# Alice Smith works at Acme, because Alice Smith and Alice Jones are the same entity.
g_same = StarLayerGraph()
g_same.bind("ex", EX)
g_same.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:aliceSmith owl:sameAs ex:aliceJones .
    ex:aliceJones ex:worksAt ex:Acme .
""", format="turtle")

show(g_same, "Before")

closed_same = g_same.infer(entailment=ENTAILMENT['OWL-RDF-Based'])

show(closed_same, "After infer(entailment=ENTAILMENT['OWL-RDF-Based'])")


Before
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:aliceJones ex:worksAt ex:Acme .

ex:aliceSmith owl:sameAs ex:aliceJones .


After infer(entailment=ENTAILMENT['OWL-RDF-Based'])
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:Acme owl:sameAs ex:Acme .

ex:aliceJones ex:worksAt ex:Acme ;
    owl:sameAs ex:aliceJones, ex:aliceSmith .

ex:aliceSmith ex:worksAt ex:Acme ;
    owl:sameAs ex:aliceJones, ex:aliceSmith .

ex:worksAt owl:sameAs ex:worksAt .



### 4.4 Running RDFS and OWL 2 RL together

`g.infer(entailment={ENTAILMENT.RDFS, ENTAILMENT['OWL-RDF-Based']})` combines both rule sets into a single closure. What `{ENTAILMENT.RDFS, ENTAILMENT['OWL-RDF-Based']}` mostly adds on top of `ENTAILMENT['OWL-RDF-Based']` alone is RDFS's *vocabulary-level* rules — universal `rdfs:Resource` typing for every term, visible below as `a owl:Thing, rdfs:Resource` on every `ex:` subject, absent from a `entailment=ENTAILMENT['OWL-RDF-Based']`-only closure of the same data.

In [8]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .

    ex:Manager rdfs:subClassOf ex:Employee .
    ex:Manager owl:equivalentClass ex:TeamLead .
    ex:alice a ex:Manager .

    ex:partOf a owl:TransitiveProperty .
    ex:SalesTeam ex:partOf ex:SalesDept .
    ex:SalesDept ex:partOf ex:AcmeCorp .
""", format="turtle")

show(g, "Before")

closed = g.infer(entailment={ENTAILMENT.RDFS, ENTAILMENT['OWL-RDF-Based']})

show(closed, "After infer(entailment={ENTAILMENT.RDFS, ENTAILMENT['OWL-RDF-Based']})")


Before
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Manager rdfs:subClassOf ex:Employee ;
    owl:equivalentClass ex:TeamLead .

ex:SalesDept ex:partOf ex:AcmeCorp .

ex:SalesTeam ex:partOf ex:SalesDept .

ex:alice a ex:Manager .

ex:partOf a owl:TransitiveProperty .




After infer(entailment={ENTAILMENT.RDFS, ENTAILMENT['OWL-RDF-Based']})
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:AcmeCorp a rdfs:Resource, owl:Thing ;
    owl:sameAs ex:AcmeCorp .

ex:Employee a rdfs:Resource, owl:Thing ;
    owl:sameAs ex:Employee .

ex:Manager a rdfs:Resource, owl:Thing ;
    rdfs:subClassOf ex:Employee, ex:Manager, ex:TeamLead ;
    owl:equivalentClass ex:Manager, ex:TeamLead ;
    owl:sameAs ex:Manager .

ex:SalesDept a rdfs:Resource, owl:Thing ;
    ex:partOf ex:AcmeCorp ;
    owl:sameAs ex:SalesDept .

ex:SalesTeam a rdfs:Resource, owl:Thing ;
    ex:partOf ex:AcmeCorp, ex:SalesDept ;
    owl:sameAs ex:SalesTeam .

ex:TeamLead a rdfs:Resource, owl:Thing ;
    rdfs:subClassOf ex:Employee, ex:Manager, ex:TeamLead ;
    owl:equivalentClass ex:Manager, ex:TeamLead ;
    owl:sameAs ex:TeamLead .

ex:alice a

### 4.5 Detecting inconsistencies

`owlrl` also checks a number of the OWL 2 RL "consistency" rules while it reasons — e.g. two classes declared `owl:disjointWith` sharing a member, `owl:sameAs` clashing with `owl:differentFrom`, or a cardinality restriction being violated. It doesn't raise an exception when one is found; instead it adds an ordinary triple into the result graph. Checking for `err:error` triples in the result is how you detect inconsistencies.

Not everything is checked this way: literal-*value* distinctness (e.g. an `owl:FunctionalProperty` given two different literal values for the same subject) is explicitly unimplemented in the current version of `owlrl` — its own source comments this as "skipped on purpose at the moment" — so that kind of inconsistency passes through silently rather than being flagged.

In [9]:
# Dog and Cat are declared disjoint, but fido is asserted to be both - a genuine inconsistency
ERR = "http://www.daml.org/2002/03/agents/agent-ont#"

g5 = StarLayerGraph()
g5.bind("ex", EX)
g5.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    ex:Dog owl:disjointWith ex:Cat .
    ex:fido a ex:Dog, ex:Cat .
""", format="turtle")

closed = g5.infer(entailment=ENTAILMENT['OWL-RDF-Based'])

# also_ns=ERR keeps the err:ErrorMessage/err:error triples visible alongside the
# ex: ones, in the show() function above.  even though their subject is a synthetic BNode rather than ex:something
show(closed, "After infer(entailment=ENTAILMENT['OWL-RDF-Based']) with inconsistency.", also_ns=ERR)


After infer(entailment=ENTAILMENT['OWL-RDF-Based']) with inconsistency.
@prefix err: <http://www.daml.org/2002/03/agents/agent-ont#> .
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .

ex:Cat owl:sameAs ex:Cat .

ex:Dog owl:disjointWith ex:Cat ;
    owl:sameAs ex:Dog .

ex:fido a ex:Cat, ex:Dog ;
    owl:sameAs ex:fido .

err:ErrorMessage owl:sameAs err:ErrorMessage .

err:error owl:sameAs err:error .

_:b0 a err:ErrorMessage ;
    err:error "Disjoint classes http://example.org/Dog and http://example.org/Cat have a common individual http://example.org/fido" .



## 5. Delta and in-place mutation

Sections 2-3 above used `infer()`'s default mode `full` to create a new graph with the inferred triples. 

The examples below demonstrate the two additional modes, `delta` and `in-place`.

### 5.1 Materializing only the delta

`infer(mode="delta")` returns only the *newly*-entailed triples. 


In [10]:
# Same as the RDFS example in section 2 above.  
# delta produces the triples that are added

g3 = StarLayerGraph()
g3.bind("ex", EX)
g3.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:alice a ex:Manager .
""", format="turtle")

closure = g3.infer(entailment=ENTAILMENT.RDFS, mode="full")
delta = g3.infer(entailment=ENTAILMENT.RDFS, mode="delta")

print("original:", len(g3), "triples")
print("full     (mode='full', default):  ", len(closure), "triples - includes the originals")
print("delta    (mode='delta'):          ", len(delta), "triples - just what's new")
show(delta, "delta only")

# the original combined with delta reproduces the full graph exactly
print("\noriginal ∪ delta == full closure:", set(closure) == set(g3) | set(delta))

original: 2 triples
full     (mode='full', default):   17 triples - includes the originals
delta    (mode='delta'):           15 triples - just what's new

delta only
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Employee a rdfs:Resource .

ex:Manager a rdfs:Resource .

ex:alice a ex:Employee, rdfs:Resource .


original ∪ delta == full closure: True


### 5.2 Mutating in place

`mode="in-place"` adds the newly-entailed triples into the original graph.  

Note: if you mutate a graph and later delete or edit triples, entailed triples created from the original data will remain.

In [11]:
g4 = StarLayerGraph()
g4.bind("ex", EX)
g4.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Manager rdfs:subClassOf ex:Employee .
    ex:alice a ex:Manager .
""", format="turtle")

print("before:", len(g4), "triples")
g4.infer(entailment=ENTAILMENT.RDFS, mode="in-place")
print("after: ", len(g4), "triples")
show(g4, "g4 after mutating in place")

# Removing the fact that justified the entailment
# doesn't retract the entailment itself - infer(mode="in-place") only ever adds triples,
# it never re-checks earlier entailments when the graph changes.
from rdflib import URIRef  # rdflib terms are accepted as input; pyoxigraph NamedNodes work equally well
RDF_TYPE = URIRef("http://www.w3.org/1999/02/22-rdf-syntax-ns#type")
alice, Manager, Employee = URIRef(EX + "alice"), URIRef(EX + "Manager"), URIRef(EX + "Employee")
g4.remove((alice, RDF_TYPE, Manager))
print("\nafter removing 'alice a Manager':")
print("alice a Manager:", (alice, RDF_TYPE, Manager) in g4, "(gone, as expected)")
print("alice a Employee:", (alice, RDF_TYPE, Employee) in g4, "(still there - stale)")

before: 2 triples
after:  17 triples

g4 after mutating in place
@prefix ex: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Employee a rdfs:Resource .

ex:Manager a rdfs:Resource ;
    rdfs:subClassOf ex:Employee .

ex:alice a ex:Employee, ex:Manager, rdfs:Resource .


after removing 'alice a Manager':
alice a Manager: False (gone, as expected)
alice a Employee: True (still there - stale)


## 6. Which entailment regime to use

- **RDF Entailment** (section 2) - narrowest, rarely used directly; cheapest to compute.
- **RDFS Entailment** (section 3) - cheap, covers the full RDFS "data" ruleset. The right default whenever your data doesn't need genuine OWL constructs.
- **OWL 2 RDF-Based Semantics**, OWL 2 RL fragment (section 4) - for graphs using OWL characteristics RDFS rewriting can't express (`owl:equivalentClass`/`owl:equivalentProperty`, property characteristics, `owl:sameAs`).
- **OWL 2 Direct Semantics** - full OWL 2 DL reasoning (disjunction, full class satisfiability) Covered in a  separate guide, [OWL 2 DL reasoning](02c-owl-dl-reasoning.ipynb), since it needs a separate reasoner (HermiT or RustDL).